# Syracuse Housing Risk Explorer
## Exploratory Data Analysis — Visualizations

**Datasets used:**
- `Syracuse_Parcel_Map__2025_Q4_.csv` — 41,263 parcels, 68 columns
- `Unfit_Properties_7837252293365347538.csv` — 335 records, 30 columns
- `Census_Tracts_in_Syracuse_NY_-_2020_9214415915210893132.csv` — 55 tracts

**Sections:**
1. Setup & Data Loading
2. Preprocessing
3. Summary Statistics
4. Visualizations (Fig 1–12)
5. Pattern Discovery


---
## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Output folder
os.makedirs('outputs/figures', exist_ok=True)

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'font.family': 'DejaVu Sans',
})

BLUE   = '#2E75B6'
RED    = '#C00000'
ORANGE = '#E36C09'
GREEN  = '#538135'
GREY   = '#7F7F7F'

print('Libraries loaded successfully.')

In [ ]:
# ── Update DATA_DIR to the folder containing your CSV files ──
DATA_DIR = '/mnt/user-data/uploads'

pm = pd.read_csv(f'{DATA_DIR}/Syracuse_Parcel_Map__2025_Q4_.csv', low_memory=False)
up = pd.read_csv(f'{DATA_DIR}/Unfit_Properties_7837252293365347538.csv', low_memory=False)
ct = pd.read_csv(f'{DATA_DIR}/Census_Tracts_in_Syracuse_NY_-_2020_9214415915210893132.csv', low_memory=False)

print(f'Parcel Map:       {pm.shape[0]:,} rows × {pm.shape[1]} cols')
print(f'Unfit Properties: {up.shape[0]:,} rows × {up.shape[1]} cols')
print(f'Census Tracts:    {ct.shape[0]:,} rows × {ct.shape[1]} cols')

---
## 2. Preprocessing

In [ ]:
# ── Parcel Map ────────────────────────────────────────────────
pm_clean = pm.copy()

# Remove 168 blank-SBL shell records
pm_clean = pm_clean[pm_clean['SBL'].astype(str).str.strip() != ''].copy()

# yr_built = 0 → NaN (vacant land placeholder, not a real build year)
pm_clean['yr_built'] = pm_clean['yr_built'].replace(0, np.nan)

# Binary vacancy flag
pm_clean['is_vacant'] = pm_clean['IPSVacant'].apply(
    lambda x: 1 if str(x).strip() in ('Residential', 'Commercial') else 0
)

# IPS condition score: '1-Worst' → 1, '5-Best' → 5, blank → NaN
def parse_condit(val):
    s = str(val).strip()
    return int(s[0]) if s and s != 'nan' else np.nan

pm_clean['condit_score'] = pm_clean['IPS_Condit'].apply(parse_condit)

# Assessed value: 0 → NaN
pm_clean['total_av_clean'] = pm_clean['total_av'].replace(0, np.nan)

# Building age (as of 2025)
pm_clean['bldg_age'] = 2025 - pm_clean['yr_built']

# Clean neighbourhood names
pm_clean['NHOOD'] = pm_clean['NHOOD'].astype(str).str.strip()

# Residential flag
residential_lu = [
    '1 Family Res', '2 Family Res', '3 Family Res',
    'Apartment', 'Converted Res', 'Multiple res',
    'Att row bldg', 'Det row bldg',
]
pm_clean['is_residential'] = pm_clean['LU_parcel'].isin(residential_lu).astype(int)

# ── Unfit Properties ─────────────────────────────────────────
up_clean = up.copy()
up_clean['violation_date']  = pd.to_datetime(up_clean['violation_date'],  errors='coerce')
up_clean['comp_open_date']  = pd.to_datetime(up_clean['comp_open_date'],  errors='coerce')
up_clean['comp_close_date'] = pd.to_datetime(up_clean['comp_close_date'], errors='coerce')

# Replace 1900-01-01 sentinel with NaT
mask_sentinel = up_clean['comp_close_date'].dt.year == 1900
up_clean.loc[mask_sentinel, 'comp_close_date'] = pd.NaT

up_clean['year']  = up_clean['violation_date'].dt.year
up_clean['month'] = up_clean['violation_date'].dt.month
up_clean['days_to_close'] = (up_clean['comp_close_date'] - up_clean['comp_open_date']).dt.days

# Unfit flag on parcel map
unfit_sbls = set(up_clean['SBL'].dropna().unique())
pm_clean['is_unfit'] = pm_clean['SBL'].isin(unfit_sbls).astype(int)

print(f'Parcel Map (clean):             {len(pm_clean):,} rows')
print(f'Unfit SBLs matched to parcels:  {pm_clean["is_unfit"].sum()}')
print(f'Parcels flagged as vacant:      {pm_clean["is_vacant"].sum():,}')
print(f'Parcels with IPS score:         {pm_clean["condit_score"].notna().sum():,}')

In [ ]:
# ── Neighbourhood-level aggregation (used in multiple figures) ─
nh_stats = (
    pm_clean[pm_clean['NHOOD'] != '']
    .groupby('NHOOD')
    .agg(
        total_parcels   = ('SBL', 'count'),
        unfit_parcels   = ('is_unfit', 'sum'),
        vacant_parcels  = ('is_vacant', 'sum'),
        residential     = ('is_residential', 'sum'),
        median_av       = ('total_av_clean', 'median'),
        median_yr_built = ('yr_built', 'median'),
        total_res_units = ('n_ResUnits', 'sum'),
    )
    .reset_index()
)
nh_stats['unfit_rate_pct']   = (nh_stats['unfit_parcels']  / nh_stats['total_parcels'] * 100).round(2)
nh_stats['vacancy_rate_pct'] = (nh_stats['vacant_parcels'] / nh_stats['total_parcels'] * 100).round(2)

print('Neighbourhood aggregation ready:', nh_stats.shape)
nh_stats.sort_values('unfit_rate_pct', ascending=False).head(10)

---
## 3. Summary Statistics

In [ ]:
# ── Parcel Map numeric summary ────────────────────────────────
num_cols = ['total_av_clean', 'land_av', 'yr_built', 'bldg_age', 'n_ResUnits', 'ACRES']
pm_clean[num_cols].describe().round(2)

In [ ]:
# ── Land use distribution ─────────────────────────────────────
lu = pm_clean['LU_parcel'].value_counts()
lu_pct = (lu / len(pm_clean) * 100).round(1)
pd.DataFrame({'count': lu, 'pct_%': lu_pct}).head(15)

In [ ]:
# ── Assessed value brackets ───────────────────────────────────
bins   = [0, 50_000, 100_000, 200_000, 500_000, 1_000_000, float('inf')]
labels = ['< $50k', '$50k–$100k', '$100k–$200k', '$200k–$500k', '$500k–$1M', '> $1M']
pm_clean['av_bracket'] = pd.cut(pm_clean['total_av_clean'], bins=bins, labels=labels, right=False)
pm_clean['av_bracket'].value_counts().sort_index()

In [ ]:
# ── Unfit Properties summary ──────────────────────────────────
print(f'Total records:       {len(up_clean)}')
print(f'Unique properties:   {up_clean["SBL"].nunique()}')
print(f'Still open:          {up_clean["comp_close_date"].isna().sum()}  ({up_clean["comp_close_date"].isna().mean()*100:.1f}%)')
print(f'Genuinely closed:    {up_clean["comp_close_date"].notna().sum()}')
print()
print('--- Complaint Type ---')
print(up_clean['complaint_type_name'].value_counts().to_string())

In [ ]:
# ── Days to close (closed cases only) ────────────────────────
closed_times = up_clean['days_to_close'].dropna()
closed_times = closed_times[closed_times >= 0]
print(f'Closed cases: {len(closed_times)}')
closed_times.describe().round(1)

---
## 4. Visualizations

### Fig 1 — Land Use Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

top_lu = pm_clean['LU_parcel'].value_counts().head(12)
colors = [BLUE if i < 3 else GREY for i in range(len(top_lu))]

ax.barh(top_lu.index[::-1], top_lu.values[::-1],
        color=colors[::-1], edgecolor='white', height=0.7)

for bar in ax.patches:
    w = bar.get_width()
    ax.text(w + 100, bar.get_y() + bar.get_height() / 2,
            f'{w:,.0f}', va='center', fontsize=9, color='#333333')

ax.set_xlabel('Number of Parcels')
ax.set_title('Fig 1 — Land Use Distribution (Top 12 Categories)')
ax.set_xlim(0, top_lu.values.max() * 1.15)
plt.tight_layout()
plt.savefig('outputs/figures/fig01_land_use.png')
plt.show()

### Fig 2 — Assessed Value Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: log-scale histogram
av_data = pm_clean['total_av_clean'].dropna()
av_data = av_data[av_data > 0]
axes[0].hist(np.log10(av_data), bins=50, color=BLUE, edgecolor='white', alpha=0.85)
axes[0].set_xlabel('log₁₀(Total Assessed Value)')
axes[0].set_ylabel('Parcel Count')
axes[0].set_title('Assessed Value Distribution (log scale)')
xticks = [4, 4.5, 5, 5.5, 6, 6.5]
axes[0].set_xticks(xticks)
axes[0].set_xticklabels([f'${10**x:,.0f}' for x in xticks], rotation=30, ha='right')

# Right: boxplot by residential type
res_types = ['1 Family Res', '2 Family Res', '3 Family Res', 'Apartment']
res_data  = pm_clean[pm_clean['LU_parcel'].isin(res_types)].copy()
res_data['total_av_log'] = np.log10(res_data['total_av_clean'].replace(0, np.nan))
res_data.boxplot(
    column='total_av_log', by='LU_parcel', ax=axes[1],
    patch_artist=True,
    boxprops=dict(facecolor=BLUE, alpha=0.6),
    medianprops=dict(color=RED, linewidth=2),
    flierprops=dict(marker='o', markersize=2, alpha=0.3),
    return_type='dict'
)
axes[1].set_xlabel('')
axes[1].set_ylabel('log₁₀(Total Assessed Value)')
axes[1].set_title('Assessed Value by Residential Type')
axes[1].set_xticklabels(res_types, rotation=15, ha='right')
plt.suptitle('')

fig.suptitle('Fig 2 — Assessed Value Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/fig02_assessed_value.png')
plt.show()

### Fig 3 — Building Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

yr_data = pm_clean['yr_built'].dropna()
yr_data = yr_data[(yr_data >= 1850) & (yr_data <= 2025)]

# Left: histogram
axes[0].hist(yr_data, bins=range(1850, 2030, 10), color=BLUE, edgecolor='white', alpha=0.85)
axes[0].axvline(yr_data.median(), color=RED, linewidth=2, linestyle='--',
                label=f'Median: {int(yr_data.median())}')
axes[0].set_xlabel('Year Built')
axes[0].set_ylabel('Parcel Count')
axes[0].set_title('Building Age Distribution (by decade)')
axes[0].legend()

# Right: decade bar chart
pm_decade = pm_clean.copy()
pm_decade['decade'] = (pm_clean['yr_built'].dropna() // 10 * 10).astype('Int64')
decade_counts = pm_decade['decade'].value_counts().sort_index()
decade_counts = decade_counts[decade_counts.index >= 1870]

axes[1].bar(
    decade_counts.index.astype(str), decade_counts.values,
    color=[RED if d >= 1980 else BLUE for d in decade_counts.index],
    edgecolor='white', alpha=0.85
)
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('Parcels Built')
axes[1].set_title('Parcels Built by Decade')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(handles=[
    mpatches.Patch(color=BLUE, alpha=0.85, label='Pre-1980 (older)'),
    mpatches.Patch(color=RED,  alpha=0.85, label='Post-1980 (newer)'),
])

fig.suptitle('Fig 3 — Building Age Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/fig03_building_age.png')
plt.show()

### Fig 4 — Unfit Properties: Complaint Types & Case Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: complaint types
ct_counts = up_clean['complaint_type_name'].value_counts().head(8)
bar_colors = [RED if i == 0 else ORANGE if i == 1 else BLUE for i in range(len(ct_counts))]
axes[0].barh(ct_counts.index[::-1], ct_counts.values[::-1],
             color=bar_colors[::-1], edgecolor='white', height=0.7)
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                 f'{int(bar.get_width())}', va='center', fontsize=9)
axes[0].set_xlabel('Number of Records')
axes[0].set_title('Complaint Type Distribution')

# Right: status pie chart
def group_status(s):
    s = str(s).strip().lower()
    if 'open' in s or 'needs' in s or 'pending' in s or 'review' in s: return 'Open / Pending'
    if 'court' in s or 'law' in s or 'baa' in s or 'collect' in s or 'judgement' in s: return 'Legal / Referred'
    if 'cert' in s: return 'Certificate Stage'
    if 'closed' in s or 'completed' in s or 'business' in s or 'denied' in s: return 'Closed / Resolved'
    return 'Other'

up_clean['status_group'] = up_clean['comp_status'].apply(group_status)
sg = up_clean['status_group'].value_counts()
wedges, texts, autotexts = axes[1].pie(
    sg.values, labels=sg.index, autopct='%1.1f%%',
    colors=[RED, ORANGE, BLUE, GREEN][:len(sg)],
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
    textprops={'fontsize': 10}
)
for at in autotexts: at.set_fontsize(9)
axes[1].set_title('Case Status Distribution')

fig.suptitle('Fig 4 — Unfit Properties: Complaint Types & Case Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/fig04_unfit_types_status.png')
plt.show()

### Fig 5 — Unfit Property Records by Year

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

yr_counts = up_clean['year'].value_counts().sort_index()
yr_counts = yr_counts[(yr_counts.index >= 2012) & (yr_counts.index <= 2025)]
bar_colors = [RED if y >= 2023 else BLUE for y in yr_counts.index]

ax.bar(yr_counts.index.astype(str), yr_counts.values,
       color=bar_colors, edgecolor='white', alpha=0.9)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Records')
ax.set_title('Fig 5 — Unfit Property Records by Year (2012–2025)')
ax.tick_params(axis='x', rotation=45)

for i, (year, val) in enumerate(yr_counts.items()):
    ax.text(i, val + 1, str(val), ha='center', fontsize=9, color='#333333')

ax.legend(handles=[mpatches.Patch(color=RED, alpha=0.85, label='2023–2025: Sharp increase')])
plt.tight_layout()
plt.savefig('outputs/figures/fig05_unfit_temporal.png')
plt.show()

### Fig 6 — Unfit Rate by Neighbourhood

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

nh_plot = nh_stats[nh_stats['unfit_parcels'] > 0].sort_values('unfit_rate_pct')
bar_colors = [
    RED    if r > 1.5 else
    ORANGE if r > 1.0 else
    BLUE
    for r in nh_plot['unfit_rate_pct']
]

ax.barh(nh_plot['NHOOD'], nh_plot['unfit_rate_pct'],
        color=bar_colors, edgecolor='white', height=0.7)
ax.axvline(1.0, color=GREY, linewidth=1, linestyle='--', alpha=0.6)

for bar in ax.patches:
    w = bar.get_width()
    ax.text(w + 0.02, bar.get_y() + bar.get_height() / 2,
            f'{w:.2f}%', va='center', fontsize=9)

ax.set_xlabel('Unfit Parcels as % of Total Neighbourhood Parcels')
ax.set_title('Fig 6 — Unfit Rate by Neighbourhood')
ax.set_xlim(0, nh_plot['unfit_rate_pct'].max() * 1.2)
ax.legend(handles=[
    mpatches.Patch(color=RED,    alpha=0.85, label='> 1.5% (High risk)'),
    mpatches.Patch(color=ORANGE, alpha=0.85, label='1.0–1.5% (Elevated)'),
    mpatches.Patch(color=BLUE,   alpha=0.85, label='< 1.0% (Moderate)'),
], loc='lower right')

plt.tight_layout()
plt.savefig('outputs/figures/fig06_unfit_rate_neighbourhood.png')
plt.show()

### Fig 7 — Vacancy Rate vs Unfit Rate (Scatter)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

nh_plot2 = nh_stats[nh_stats['NHOOD'] != ''].copy()
sc = ax.scatter(
    nh_plot2['vacancy_rate_pct'],
    nh_plot2['unfit_rate_pct'],
    s=nh_plot2['total_parcels'] / 10,
    c=nh_plot2['median_av'].fillna(0),
    cmap='RdYlGn_r',
    alpha=0.8,
    edgecolors='grey',
    linewidths=0.5
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Median Assessed Value ($)', fontsize=10)

# Label high-risk neighbourhoods
for _, row in nh_plot2[nh_plot2['unfit_rate_pct'] > 1.0].iterrows():
    ax.annotate(
        row['NHOOD'],
        (row['vacancy_rate_pct'], row['unfit_rate_pct']),
        textcoords='offset points', xytext=(6, 3),
        fontsize=8, color='#333333'
    )

ax.axhline(1.0, color=GREY, linestyle='--', linewidth=1, alpha=0.5)
ax.axvline(5.0, color=GREY, linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Vacancy Rate (%)')
ax.set_ylabel('Unfit Rate (%)')
ax.set_title('Fig 7 — Vacancy Rate vs Unfit Rate by Neighbourhood\n(bubble size = parcel count, colour = median assessed value)')

plt.tight_layout()
plt.savefig('outputs/figures/fig07_vacancy_vs_unfit.png')
plt.show()

### Fig 8 — Assessed Value: Unfit vs Non-Unfit Parcels

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

av_unfit     = pm_clean.loc[(pm_clean['is_unfit'] == 1) & pm_clean['total_av_clean'].notna(), 'total_av_clean']
av_non_unfit = pm_clean.loc[(pm_clean['is_unfit'] == 0) & pm_clean['total_av_clean'].notna(), 'total_av_clean']
av_unfit     = av_unfit[(av_unfit > 0) & (av_unfit < 2_000_000)]
av_non_unfit = av_non_unfit[(av_non_unfit > 0) & (av_non_unfit < 2_000_000)]

ax.hist(av_non_unfit / 1000, bins=60, alpha=0.55, color=BLUE,
        label=f'Non-Unfit  (n={len(av_non_unfit):,})', density=True)
ax.hist(av_unfit     / 1000, bins=60, alpha=0.70, color=RED,
        label=f'Unfit  (n={len(av_unfit):,})', density=True)

ax.set_xlabel('Total Assessed Value ($000s)')
ax.set_ylabel('Density')
ax.set_title('Fig 8 — Assessed Value: Unfit vs Non-Unfit Parcels (< $2M)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/fig08_av_unfit_vs_normal.png')
plt.show()

### Fig 9 — Building Age: Unfit vs Non-Unfit Parcels

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

age_unfit     = pm_clean.loc[(pm_clean['is_unfit'] == 1) & pm_clean['yr_built'].notna(), 'yr_built']
age_non_unfit = pm_clean.loc[(pm_clean['is_unfit'] == 0) & pm_clean['yr_built'].notna(), 'yr_built']
age_unfit     = age_unfit[(age_unfit >= 1850) & (age_unfit <= 2025)]
age_non_unfit = age_non_unfit[(age_non_unfit >= 1850) & (age_non_unfit <= 2025)]

ax.hist(age_non_unfit, bins=range(1850, 2030, 5), alpha=0.55, color=BLUE,
        label=f'Non-Unfit  (n={len(age_non_unfit):,})', density=True)
ax.hist(age_unfit,     bins=range(1850, 2030, 5), alpha=0.70, color=RED,
        label=f'Unfit  (n={len(age_unfit):,})', density=True)

ax.axvline(age_unfit.median(),     color=RED,  linestyle='--', linewidth=1.5,
           label=f'Unfit median: {int(age_unfit.median())}')
ax.axvline(age_non_unfit.median(), color=BLUE, linestyle='--', linewidth=1.5,
           label=f'Non-Unfit median: {int(age_non_unfit.median())}')

ax.set_xlabel('Year Built')
ax.set_ylabel('Density')
ax.set_title('Fig 9 — Building Age: Unfit vs Non-Unfit Parcels')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('outputs/figures/fig09_age_unfit_vs_normal.png')
plt.show()

### Fig 10 — Land Use: All Parcels vs Unfit Properties

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

all_lu   = pm_clean['LU_parcel'].value_counts().head(10)
unfit_lu = pm_clean[pm_clean['is_unfit'] == 1]['LU_parcel'].value_counts().head(10)

all_pct   = (all_lu   / len(pm_clean) * 100).round(1)
unfit_pct = (unfit_lu / pm_clean['is_unfit'].sum() * 100).round(1)

axes[0].barh(all_pct.index[::-1], all_pct.values[::-1],
             color=BLUE, edgecolor='white', height=0.7)
axes[0].set_title('All Parcels — Land Use %')
axes[0].set_xlabel('% of Total')
for bar in axes[0].patches:
    axes[0].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                 f'{bar.get_width():.1f}%', va='center', fontsize=9)

axes[1].barh(unfit_pct.index[::-1], unfit_pct.values[::-1],
             color=RED, edgecolor='white', height=0.7)
axes[1].set_title('Unfit Properties — Land Use %')
axes[1].set_xlabel('% of Unfit')
for bar in axes[1].patches:
    axes[1].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                 f'{bar.get_width():.1f}%', va='center', fontsize=9)

fig.suptitle('Fig 10 — Land Use: All Parcels vs Unfit Properties', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/fig10_landuse_comparison.png')
plt.show()

### Fig 11 — Unfit Records: Neighbourhood × Year Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

# Join neighbourhood from parcel map
up_nh = up_clean.merge(
    pm_clean[['SBL', 'NHOOD']].drop_duplicates(subset='SBL'),
    on='SBL', how='left'
)
up_nh = up_nh[
    up_nh['NHOOD'].notna() &
    (up_nh['NHOOD'] != '') &
    up_nh['year'].notna() &
    (up_nh['year'] >= 2018)
]

pivot = up_nh.pivot_table(
    index='NHOOD', columns='year', values='SBL', aggfunc='count', fill_value=0
)
# Keep top 20 neighbourhoods by total records
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).head(20).index]

sns.heatmap(
    pivot, ax=ax,
    cmap='YlOrRd',
    annot=True, fmt='d',
    linewidths=0.5,
    cbar_kws={'label': 'Unfit Records'},
    annot_kws={'size': 9}
)
ax.set_title('Fig 11 — Unfit Property Records by Neighbourhood × Year (Top 20, 2018–2025)')
ax.set_xlabel('Year')
ax.set_ylabel('Neighbourhood')
plt.tight_layout()
plt.savefig('outputs/figures/fig11_heatmap_nh_year.png')
plt.show()

### Fig 12 — Correlation Matrix: Key Parcel Features

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

corr_cols = ['total_av_clean', 'yr_built', 'bldg_age', 'n_ResUnits',
             'ACRES', 'is_vacant', 'is_unfit', 'is_residential']
corr_df = pm_clean[corr_cols].dropna(subset=['total_av_clean', 'yr_built'])
corr_matrix = corr_df.corr().round(2)

rename_map = {
    'total_av_clean': 'Assessed Value',
    'yr_built':       'Year Built',
    'bldg_age':       'Building Age',
    'n_ResUnits':     'Res. Units',
    'ACRES':          'Lot Size (acres)',
    'is_vacant':      'Vacant Flag',
    'is_unfit':       'Unfit Flag',
    'is_residential': 'Residential',
}
corr_matrix = corr_matrix.rename(columns=rename_map, index=rename_map)

# Lower triangle only
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, ax=ax,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, mask=mask, square=True,
    cbar_kws={'shrink': 0.8},
    annot_kws={'size': 9}
)
ax.set_title('Fig 12 — Correlation Matrix: Key Parcel Features')
plt.tight_layout()
plt.savefig('outputs/figures/fig12_correlation_matrix.png')
plt.show()

---
## 5. Pattern Discovery

In [ ]:
# ── Pattern 1: Land use over-representation in unfit ─────────
lu_all   = pm_clean['LU_parcel'].value_counts()
lu_unfit = pm_clean[pm_clean['is_unfit'] == 1]['LU_parcel'].value_counts()

lu_compare = pd.DataFrame({
    'all_%':   (lu_all   / len(pm_clean) * 100).round(1),
    'unfit_%': (lu_unfit / pm_clean['is_unfit'].sum() * 100).round(1),
}).dropna()
lu_compare['ratio'] = (lu_compare['unfit_%'] / lu_compare['all_%']).round(2)

print('PATTERN 1 — Land use types most over-represented among unfit properties')
lu_compare[lu_compare['ratio'] != np.inf].sort_values('ratio', ascending=False).head(10)

In [ ]:
# ── Pattern 2: Building age — unfit vs non-unfit ─────────────
print('PATTERN 2 — Building Age Comparison')
print(f"Median yr_built (unfit):       {pm_clean.loc[pm_clean['is_unfit']==1,'yr_built'].median():.0f}")
print(f"Median yr_built (non-unfit):   {pm_clean.loc[pm_clean['is_unfit']==0,'yr_built'].median():.0f}")
print(f"% unfit built before 1930:     {(pm_clean.loc[pm_clean['is_unfit']==1,'yr_built'] < 1930).mean()*100:.1f}%")
print(f"% non-unfit built before 1930: {(pm_clean.loc[pm_clean['is_unfit']==0,'yr_built'] < 1930).mean()*100:.1f}%")

In [ ]:
# ── Pattern 3: Vacancy ↔ Unfit co-occurrence ─────────────────
print('PATTERN 3 — Vacancy & Unfit Co-Occurrence')
print(pd.crosstab(pm_clean['is_vacant'], pm_clean['is_unfit'],
                  rownames=['Vacant'], colnames=['Unfit']))

vacant_unfit     = pm_clean[pm_clean['is_vacant']==1]['is_unfit'].mean() * 100
non_vacant_unfit = pm_clean[pm_clean['is_vacant']==0]['is_unfit'].mean() * 100
print(f"\nUnfit rate — vacant parcels:     {vacant_unfit:.2f}%")
print(f"Unfit rate — non-vacant parcels: {non_vacant_unfit:.2f}%")
print(f"Multiplier: {vacant_unfit / non_vacant_unfit:.1f}×")

In [ ]:
# ── Pattern 4: Low-value parcel concentration by neighbourhood ─
print('PATTERN 4 — Top 10 Neighbourhoods: Rate of Parcels Assessed < $50k')
low_av_rate = (
    pm_clean[pm_clean['NHOOD'] != '']
    .groupby('NHOOD')
    .apply(lambda df: (df['total_av_clean'] < 50_000).mean() * 100)
    .sort_values(ascending=False)
    .round(1)
)
print(low_av_rate.head(10).to_string())

In [ ]:
# ── Pattern 5: Out-of-state ownership ────────────────────────
print('PATTERN 5 — Out-of-State Ownership (Unfit Properties)')
oos = up_clean[up_clean['owner_state'].notna() & (up_clean['owner_state'].str.strip() != '')]
oos_rate = (oos['owner_state'] != 'NY').mean() * 100
print(f"Out-of-state ownership rate: {oos_rate:.1f}%")
print(oos['owner_state'].value_counts())

In [ ]:
# ── Pattern 6: Dual-risk neighbourhoods ──────────────────────
print('PATTERN 6 — Dual-Risk Neighbourhoods (Vacancy > 5% AND Unfit Rate > 1%)')
dual_risk = nh_stats[
    (nh_stats['vacancy_rate_pct'] > 5) &
    (nh_stats['unfit_rate_pct']   > 1)
][['NHOOD', 'total_parcels', 'unfit_rate_pct', 'vacancy_rate_pct', 'median_av']]
dual_risk.sort_values('unfit_rate_pct', ascending=False)

In [ ]:
# ── Pattern 7: Case resolution gap ───────────────────────────
print('PATTERN 7 — Case Resolution Gap')
total  = len(up_clean)
open_  = up_clean['comp_close_date'].isna().sum()
closed = up_clean['comp_close_date'].notna().sum()
median_days = up_clean['days_to_close'].dropna()
median_days = median_days[median_days >= 0].median()

print(f"Total records:     {total}")
print(f"Open / unresolved: {open_}  ({open_/total*100:.1f}%)")
print(f"Closed:            {closed}  ({closed/total*100:.1f}%)")
print(f"Median days to close: {median_days:.0f} days (~{median_days/30:.1f} months)")

---
*End of EDA Notebook — Syracuse Housing Risk Explorer*